# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>

# <font color="#003660">Session 5: Retrieval-Augmented Generation</font>

# <font color="#003660">Answering Questions with Documents</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... can answer questions about a document by putting it into the prompt, <br>
        ... know what the model sees when you send a document with a prompt template, <br>
        ... can build the same with LangChain's prompt templates and chains, <br>
        ... know why this stops working for long documents, and why we need RAG.
    </font>
</div>
</p>

Inspired by these excellent sources:

* [HuggingFace (2024): NLP Course](https://huggingface.co/learn/nlp-course/)
* [Prompt Engineering Guide](https://www.promptingguide.ai/)

Documentation for this notebook:

* [LangChain Docs: ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)
* [LangChain Docs: Build a RAG agent](https://docs.langchain.com/oss/python/langchain/rag)
* [LangChain API Reference](https://reference.langchain.com/python/)
* [AI-Gateway of Paderborn University](https://hilfe.uni-paderborn.de/AI-Gateway)

## Where We Are

* **4.03:** the OpenAI-compatible API with LangChain's `ChatOpenAI`, and how to choose where the model runs.
* **4.04:** structured prompts with `###` blocks and delimiters.
* **Now:** give the model a **document** and let it answer questions about it.

This is the first step towards **retrieval-augmented generation (RAG)**, the topic of this session.

## Setup

The same setup as in 4.03: run the next cell and choose where the model runs.

| `provider` | Where the model runs | `model_name` | Good to know |
|---|---|---|---|
| `"ollama"` | **Ollama** on your computer | `qwen3.5:9b`, `qwen3.5:4b`, `qwen3.5:2b` | free, no account; install [Ollama](https://ollama.com/download) first, then download the model once in a terminal: `ollama pull qwen3.5:9b` (6.6 GB; 4b: 3.3 GB; 2b: 2.7 GB); without a GPU, it runs on the CPU, which is slower |
| `"colab"` | **Ollama on Google Colab** | the same three | free, needs a Google account; the cell installs Ollama in the Colab runtime and downloads the model, as in 4.01; choose a GPU runtime first (Runtime > Change runtime type > T4 GPU) |
| `"huggingface"` | **Hugging Face endpoints** of the course | the same three, one endpoint each | key and addresses from the course; if an endpoint was asleep, the first request takes 1 to 2 minutes |
| `"ai-gateway"` | **AI-Gateway** of Paderborn University | `gwdg.qwen3.8-27b-no-think` (thinking off), `gwdg.qwen3.8-27b` (thinking on) | your university key (see [setup/SETUP_AI_GATEWAY.md](https://github.com/olivermueller/amlta-2026/blob/main/setup/SETUP_AI_GATEWAY.md)); only in the university network or the VPN, not on Colab; about 10 requests per minute |
| `"other"` | **another** OpenAI-compatible API | the name your provider uses | base URL and key in `.env` (`OTHER_BASE`, `OTHER_KEY`) |

Keys and addresses come from the `.env` file in the repo folder (see `.env.example`). Without a `.env` file, for example on Colab, the cell asks for them.

In [ ]:
%pip install -q langchain langchain-openai python-dotenv

In [ ]:
# Run this cell: it only defines setup_model(). We call it before every ChatOpenAI, with the provider and the model.
import getpass
import json
import os
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request

from dotenv import load_dotenv

load_dotenv()  # reads the .env file if there is one: this folder first, then the folders above (the repo folder)
IN_COLAB = "google.colab" in sys.modules

PROVIDERS = ["ollama", "colab", "huggingface", "ai-gateway", "other"]
QWEN = {  # model_name: (download size in Ollama, the name of its endpoint address in .env, the model on Hugging Face)
    "qwen3.5:9b": ("6.6 GB", "HF_GATEWAY_BASE_9B", "unsloth/Qwen3.5-9B-GGUF"),
    "qwen3.5:4b": ("3.3 GB", "HF_GATEWAY_BASE_4B", "unsloth/Qwen3.5-4B-GGUF"),
    "qwen3.5:2b": ("2.7 GB", "HF_GATEWAY_BASE_2B", "unsloth/Qwen3.5-2B-GGUF"),
}
GATEWAY_QWEN = {  # on the AI-Gateway, the model name decides about thinking
    "gwdg.qwen3.8-27b-no-think": "thinking off",
    "gwdg.qwen3.8-27b": "thinking on",
}


def value(env_name, question, secret=False):
    """Takes the value from .env, or asks for it (keys are typed hidden)."""
    found = os.getenv(env_name)
    if found:
        return found.strip()
    answer = getpass.getpass(question).strip() if secret else input(question).strip()
    os.environ[env_name] = answer  # asked once: the next call of setup_model() takes it from here
    return answer


def start_ollama(model, size, colab=False):
    """Makes sure that the Ollama server runs and has the model. Downloads the model only on Colab."""
    url = (os.getenv("OLLAMA_BASE") or "http://localhost:11434").rstrip("/")

    def running():
        try:
            urllib.request.urlopen(url, timeout=2)
            return True
        except OSError:
            return False

    if colab and shutil.which("ollama") is None:
        # the same steps as in 4.01: the installer needs pciutils and zstd
        print("Installing Ollama in the Colab runtime (about a minute) ...")
        subprocess.run("sudo apt-get install -y -qq pciutils zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null",
                       shell=True, check=True)
    if not running():
        if shutil.which("ollama") is None:
            raise RuntimeError("Ollama is not installed. Install it from https://ollama.com/download and run this cell again.")
        # on Colab, the installer may say that the server is running, but it is not; so we start it ourselves (as in 4.01)
        print("Starting the Ollama server ...")
        ollama_log = open("ollama.log", "w")  # a pipe that nobody reads could fill up and block the server
        subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
        for _ in range(30):
            if running():
                break
            time.sleep(1)
        else:
            raise RuntimeError("The Ollama server did not start. Look into ollama.log.")

    installed = [m["name"] for m in json.load(urllib.request.urlopen(url + "/api/tags"))["models"]]
    if model not in installed and not colab:  # on your computer, you download the model yourself, once
        raise RuntimeError(f"{model} is not downloaded yet. Run this command in a terminal (it downloads {size}), "
                           f"then run this cell again:\n\n    ollama pull {model}\n")
    if model not in installed:
        print(f"WARNING: {model} is not here yet. It is downloaded now ({size}), this can take several minutes.")
        request = urllib.request.Request(url + "/api/pull", data=json.dumps({"model": model}).encode())
        shown = -1
        for line in urllib.request.urlopen(request):
            status = json.loads(line)
            if "error" in status:
                raise RuntimeError(status["error"])
            if status.get("total"):
                percent = int(100 * status.get("completed", 0) / status["total"]) // 10 * 10
                if percent != shown:
                    print(f"  {percent} %", end="  ")
                    shown = percent
        print("\nDownload finished.")
    return url + "/v1/"


def wait_until_awake(base_url, api_key):
    """A sleeping Hugging Face endpoint answers 503 until it has started; we wait for it."""
    request = urllib.request.Request(base_url + "/models", headers={"Authorization": f"Bearer {api_key}"})
    for attempt in range(40):  # up to about 6 minutes
        try:
            urllib.request.urlopen(request, timeout=10)
            return
        except urllib.error.HTTPError as error:
            if error.code == 403:
                raise RuntimeError("The endpoint answers 403: it is paused (the course endpoints only run during the seminar), "
                                   "or your key has no access to it. Choose another provider, or check HF_GATEWAY_KEY and the address in .env.")
            if error.code != 503:
                raise RuntimeError(f"The endpoint answers {error.code}. Check the key and the address.")
        except OSError:
            pass
        if attempt == 0:
            print("The endpoint is asleep and starts now. This takes a few minutes ...")
        time.sleep(8)
    raise RuntimeError("The endpoint did not wake up. Try again in a few minutes.")


def setup_model(model):
    """Checks {"model_name": ..., "provider": ...}, sets the address and the key, and prints what to keep in mind.
    Returns base_url, api_key and the model name for ChatOpenAI."""
    name = model.get("model_name", "").strip()
    provider = model.get("provider", "").strip().lower()
    if provider not in PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}. Choose one of: {', '.join(PROVIDERS)}.")
    if provider in ("ollama", "colab", "huggingface") and name not in QWEN:
        raise ValueError(f"The provider {provider!r} has these models: {', '.join(QWEN)}. Not {name!r}.")
    notes = []

    if provider in ("ollama", "colab"):
        if provider == "ollama" and IN_COLAB:
            raise ValueError("This notebook runs on Google Colab: use the provider 'colab'.")
        if provider == "colab" and not IN_COLAB:
            raise ValueError("The provider 'colab' only works on Google Colab. On your own computer, use 'ollama'.")
        if provider == "colab" and subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode != 0:
            notes.append("This Colab runtime has no GPU: answers take long. Better: Runtime > Change runtime type > T4 GPU, then run the notebook again.")
        if provider == "ollama":
            notes.append("The model runs on this computer. Without a GPU, it runs on the CPU, and answers take longer.")
        base_url, api_key = start_ollama(name, QWEN[name][0], colab=(provider == "colab")), "ollama"  # Ollama needs no key
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): Ollama returns only one.")

    elif provider == "huggingface":
        _, env_name, name = QWEN[name]  # name: the model on Hugging Face
        base_url = value(env_name, f"Paste the address of the Hugging Face endpoint for {model['model_name']}: ").rstrip("/")
        if "<" in base_url:
            raise ValueError(f"The address for {model['model_name']} is still a placeholder in .env ({env_name}). "
                             "Paste the address from the course there, or choose another model or provider.")
        if not base_url.endswith("/v1"):
            base_url += "/v1"  # the OpenAI-compatible API of the endpoint
        api_key = value("HF_GATEWAY_KEY", "Paste your Hugging Face key (for a public endpoint, just press Enter): ", secret=True)
        api_key = api_key or "none"  # a public endpoint needs no key, but the client needs some text
        wait_until_awake(base_url, api_key)
        notes.append('Thinking: switch it with "reasoning_effort" in the extra_body of ChatOpenAI ("none" = off).')
        notes.append("Several answers per request (n > 1): the endpoint refuses them (HTTP 400).")

    elif provider == "ai-gateway":
        if IN_COLAB:
            notes.append("Colab runs outside the university network and usually cannot reach the AI-Gateway.")
        base_url = os.getenv("AI_GATEWAY_BASE") or "https://ai-gateway.uni-paderborn.de/v1"
        api_key = value("AI_GATEWAY_KEY", "Paste your AI-Gateway key: ", secret=True)
        if name not in GATEWAY_QWEN:
            notes.append(f"{name!r} is not one of the Qwen models of the course ({', '.join(GATEWAY_QWEN)}). "
                         "llm.root_client.models.list() lists the models of your account.")
        notes.append("Thinking cannot be switched with reasoning_effort here (the gateway ignores it): the model name decides. "
                     "gwdg.qwen3.8-27b-no-think = off, gwdg.qwen3.8-27b = on.")
        notes.append("Only in the university network or the VPN. About 10 requests per minute: if a cell pauses, the client waits and continues.")

    else:  # another OpenAI-compatible API
        if not name:
            raise ValueError("Give the model name of your provider as model_name.")
        base_url = value("OTHER_BASE", "Base URL of the API (e.g. https://api.openai.com/v1/): ")
        api_key = value("OTHER_KEY", "Your API key: ", secret=True)
        notes.append("Check in the documentation of your provider how thinking is switched; "
                     "the extra_body of ChatOpenAI may need other fields than reasoning_effort.")

    print(f"Provider: {provider} | model: {name}")  # the key is never printed
    print("Keep in mind:")
    for note in notes:
        print("  -", note)
    return base_url, api_key, name

**Before every `ChatOpenAI`, we call `setup_model()`** with the provider and the model, for example `setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})`. It checks whether the combination exists, sets the address and the key, and prints what to keep in mind. It returns three values, which we split directly into `BASE_URL`, `API_KEY`, and `MODEL_NAME`. To switch the model, change the dict there.

In [ ]:
import os
import time
import urllib.request

from langchain_openai import ChatOpenAI

BASE_URL, API_KEY, MODEL_NAME = setup_model({"model_name": "qwen3.5:9b", "provider": "huggingface"})  # provider and model: see the table in the setup
llm = ChatOpenAI(
    model=MODEL_NAME,  # on the AI-Gateway, the model decides about thinking: gwdg.qwen3.8-27b-no-think (off) or gwdg.qwen3.8-27b (on)
    base_url=BASE_URL,
    api_key=API_KEY,
    temperature=0,
    extra_body={
        "max_tokens": 512,  # the token limit (here, because Ollama ignores max_completion_tokens, the name that ChatOpenAI uses)
        "reasoning_effort": "none",  # thinking off; "low", "medium", or "high" switches it on (Ollama, Hugging Face endpoint)
    },
)

## Part 1: A Question the Model Cannot Know

**In short: a model only knows what was in its training data, and what is in the prompt.**

Below is the travel policy of a company. The company and the policy are made up, so the model has never seen them.

In [ ]:
POLICY = """LUMEN ANALYTICS GMBH - TRAVEL POLICY (valid from 1 January 2026)
This is a fictional document for teaching purposes.

1. Approval: Every business trip must be approved by your team lead at least 5 working days in advance. Trips abroad also need the approval of the head of department.
2. Train or plane: Within Germany, take the train in 2nd class. For train rides longer than 4 hours, you may book 1st class. Flights within Germany are only allowed if the train takes more than 6 hours.
3. Hotels: The company pays up to 110 euros per night. In Munich, Frankfurt, and Hamburg, the limit is 150 euros per night. Breakfast is included in the limit.
4. Meals: For each full travel day, you get a daily allowance of 28 euros. For the day of arrival and the day of departure, you get 14 euros each.
5. Taxis: Taxis are only paid between 10 pm and 6 am, or if you carry equipment that weighs more than 15 kg.
6. Expense report: Submit all receipts in the expense tool SpendLog within 30 days after the trip. Receipts submitted later are not reimbursed."""

First, we ask the model *without* the document:

In [ ]:
question = "I am staying two nights in a hotel in Munich. How much does the company pay for the hotel at most?"

response = llm.invoke([{"role": "user", "content": question}])
print(response.content)

The model cannot know the answer. In the best case, it says so. In the worst case, it invents a plausible number.

## Part 2: The Document Goes into the Prompt

**In short: paste the document into the prompt, and tell the model to answer only from it.**

We use what you learned in 4.04:

* a **system prompt** with the rules for every answer,
* a **prompt template** with `###` blocks, and the document between `<document>` tags.

In [ ]:
SYSTEM_PROMPT = """You answer questions about a document.
Use only the information in the document. If the document does not contain the answer, say that you don't know."""

USER_TEMPLATE = """### Document
<document>
{document}
</document>

### Question
{question}"""


def build_messages(document, question):
    """Fills the template and returns the list of messages for the chat API."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(document=document, question=question)},
    ]


messages = build_messages(POLICY, question)

### What the model sees

In 4.02, Ollama's own API showed us the finished text. The OpenAI-compatible API does not: the provider applies the chat template on the server.

So we build a **mockup**: a small function that writes our messages into the chat template of the Qwen models, the way you saw it in 4.02. The real template of your model may add a few details, but the structure is the same.

In [ ]:
def what_the_model_sees(messages):
    """A mockup: writes the messages into the Qwen chat template from 4.02.
    The API hides the real template, so this is our reconstruction."""
    text = ""
    for message in messages:
        text += f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>\n"
    text += "<|im_start|>assistant\n"  # the open assistant header: here the model starts writing
    print(text)


what_the_model_sees(messages)

Everything is one long text again: the system prompt, the document, the question, and the open assistant header at the end. The model continues this text, as always.

Now we send the same messages to the model:

In [ ]:
response = llm.invoke(messages)
print(response.content)
print(f"\n[{response.usage_metadata["input_tokens"]} prompt tokens, {response.usage_metadata["output_tokens"]} generated tokens]")

With the document in the prompt, the model can answer: 2 nights × 150 euros = 300 euros.

### Your turn

1. Ask three questions of your own about the policy. One of them should have **no** answer in the document, for example "Can I bring my dog on a business trip?".
2. Does the model say that it doesn't know? If not, change `SYSTEM_PROMPT` and try again.

In [ ]:
# TODO: ask your own questions
my_questions = [
    "TODO",
    "TODO",
    "TODO",
]

for my_question in my_questions:
    response = llm.invoke(build_messages(POLICY, my_question))
    print(my_question)
    print("->", response.content, end="\n\n")

## Part 3: Prompt Templates and Chains

**In short: LangChain gives you building blocks that you can chain.**

So far, we used one building block of [LangChain](https://docs.langchain.com/): the chat model `ChatOpenAI` (4.03). Two more make the code shorter, and we use all three for the rest of this session and in the next ones (RAG, tools, agents).

| Building block | What it does | Example |
|---|---|---|
| **Chat model** | sends messages to a model and returns the answer | `ChatOpenAI` |
| **Prompt template** | fills placeholders and returns the messages | `ChatPromptTemplate` |
| **Output parser** | turns the answer into what your code needs | `StrOutputParser` |

### The prompt template in LangChain

Our template from Part 2 works in LangChain with almost no changes. `ChatPromptTemplate` takes a list of `(role, text)` pairs with placeholders. (LangChain calls the user role `human`.)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", USER_TEMPLATE),
])

prompt_value = prompt.invoke({"document": POLICY, "question": question})
prompt_value.to_messages()

The template returns the same messages as `build_messages()`. Our mockup shows that the model sees exactly the same text as in Part 2:

In [ ]:
from langchain_core.messages import convert_to_openai_messages

what_the_model_sees(convert_to_openai_messages(prompt_value.to_messages()))

### Chaining the blocks

The `|` operator connects the blocks into a **chain**. The output of one block is the input of the next:

1. `prompt` fills the template and returns the messages.
2. `llm` sends them to the model and returns an `AIMessage`.
3. `StrOutputParser()` takes the text out of the `AIMessage`.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

chain.invoke({"document": POLICY, "question": question})

Same prompt, same model, same answer. LangChain only saves you the plumbing, and you can swap every block: a different model, a different template, a different parser.

### Your turn

Change the chain so that the model answers **in German** and names the **number of the rule** it used, for example "(Regel 3)".

Change only the prompt template, not the line with the chain. Test it with two questions.

In [ ]:
# TODO: a new system prompt; the user template stays the same
german_system_prompt = """TODO"""

german_prompt = ChatPromptTemplate.from_messages([
    ("system", german_system_prompt),
    ("human", USER_TEMPLATE),
])
german_chain = german_prompt | llm | StrOutputParser()

print(german_chain.invoke({"document": POLICY, "question": question}))

## Part 4: When the Document Gets Long

**In short: putting the whole document into every prompt does not scale.**

Our policy has a few hundred tokens. Real documents are much longer. In the next notebooks, we work with two Wikipedia articles: *Game of Thrones* and *How I Met Your Mother*.

In [ ]:
# downloads the two articles as Markdown, if they are not here yet
REPO_URL = "https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_05/"
for path in ["markdown_documents/Game_of_Thrones.md", "markdown_documents/How_I_Met_Your_Mother.md"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO_URL + path, path)

articles = {}
for path in ["markdown_documents/Game_of_Thrones.md", "markdown_documents/How_I_Met_Your_Mother.md"]:
    with open(path, encoding="utf-8") as file:
        articles[path] = file.read()
    # rule of thumb: one token is about 4 characters of English text
    print(f"{path}: {len(articles[path]):,} characters, about {len(articles[path]) // 4:,} tokens")

Let's send the whole *How I Met Your Mother* article with one question. (This takes a while and uses a good part of your token budget, so run it only once.)

In [ ]:
himym = articles["markdown_documents/How_I_Met_Your_Mother.md"]

start = time.time()
response = llm.invoke(build_messages(himym, "Who plays Ted Mosby?"))
print(response.content)
print(f"\n[{response.usage_metadata["input_tokens"]:,} prompt tokens, {time.time() - start:.0f} seconds]")

The answer is right, but look at the price: tens of thousands of prompt tokens for a one-line answer, and **every** question costs the same again.

Compare the real number of prompt tokens with our estimate from above. The estimate is too low: links, numbers, and Markdown need more tokens than normal English text. The "4 characters per token" rule is only a rough guide.

Three problems:

* **Cost and time:** you pay for the whole document with every question.
* **Limits:** a collection of documents quickly gets larger than the context window of any model.
* **Lost in the middle:** models use information in the middle of a long context worse than information at the beginning or the end ([Liu et al., 2024](https://doi.org/10.1162/tacl_a_00638)).

**The idea of RAG:** do not send the whole document. First **retrieve** only the few pieces that are relevant for the question, and then let the model **generate** the answer from them. That is the next notebook.

## Questions?

Anything unclear about documents in prompts, the template mockup, or LangChain? Ask now.

## What You Learned

* A model only knows its training data and what is in the prompt. For your own documents, the document has to go into the prompt.
* A **system prompt** plus a **template** with `###` blocks and `<document>` tags gives the model the document, the question, and the rules.
* What the model sees is still **one long text**: the API hides the template, so we built a mockup.
* **LangChain** wraps the API in building blocks: `ChatOpenAI` (model), `ChatPromptTemplate` (template), `StrOutputParser` (parser). The `|` operator chains them.
* Whole documents in every prompt are expensive, limited by the context window, and get "lost in the middle". **RAG** sends only the relevant pieces.

Next: **5.02 RAG Basics**. We build retrieval-augmented generation from scratch.